# Climate Data Cleaning: ISU Climate Data

Cleans the IEM/ISU daily climate summary (one row per station-day) into a tidy
table keyed on `station` + `day`, ready to spatially-and-temporally join onto
the EPA water-quality samples in the merge step.

**Input:**  `data/tabular/01_raw/climate/isu-climate.csv`
**Output:** `data/tabular/02_clean/climate/isu-climate-clean.csv`

**Pipeline**
1. Load the raw daily summary.
2. **Fix types & join key** — parse `day`, coerce numerics, drop rows with no
   `station`/`day` (they can never join), and enforce one row per
   `(station, day)` so the downstream left-join can't fan out.
3. **Range-validate** every measurement against physical bounds (RH 0–100,
   wind direction 0–360, plausible Iowa temperatures, non-negative precip), nulling
   impossible values instead of letting them reach the models.
4. **Cross-field consistency** — null pairs where `min > max` (e.g. a min temp
   above the day's max temp is a recording error).
5. **Missing values** — snow/snow-depth default to 0 (event variables, only
   reported when present); precipitation is left `NaN` (see below).
6. Sanity-check and save.

**Why these changes matter** — the previous version (a) pointed at the old
`data/tabular/climate/{raw,clean}` paths that no longer exist, so it could not
run; (b) ran `dropna(subset=['max_temp_f','min_temp_f','precip_in'])`, which
discarded **117k of 221k rows (53%)** — almost entirely because `precip_in` was
missing. That missingness is *per-station*: roughly half the stations have no
precip sensor and report it ~20% of the time, while the other half report it
always. Dropping those rows threw away **113k rows that had perfectly good
temperature, humidity and wind data**; filling them with 0 (as the old code did
for snow) would instead fabricate "no rain" for half the network. Keeping the
rows and leaving precip `NaN` is the honest choice and roughly **doubles**
climate coverage available to the join. The old pipeline also did no range or
consistency checking at all.

> **Path note:** this targets the migrated `01_raw`/`02_clean` layout. The merge
> step (`src/03_merge/merge_epa_climate.py`) and README still read
> `data/tabular/climate/clean/isu-climate-clean.csv` and should be updated to
> point at `02_clean/climate/`. The `station` + `day` output contract is
> unchanged, so only the path needs updating there.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies
    (repo root vs. the notebook folder), so resolving paths relative to a
    fixed number of "../" is fragile — that fragility is exactly why the old
    notebook's hard-coded '../../../../data/...' paths broke after the layout
    migration. Searching upward for a sentinel makes the notebook runnable
    from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "climate"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "climate"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/climate
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/climate


## Step 1 — Load

In [2]:
df = pd.read_csv(RAW_DIR / "isu-climate.csv")
n_raw = len(df)
print(f"Loaded {n_raw:,} station-day rows across {df['station'].nunique()} stations")
df.head()

Loaded 221,559 station-day rows across 62 stations


,station,day,max_temp_f,min_temp_f,max_dewpoint_f,min_dewpoint_f,precip_in,avg_wind_speed_kts,avg_wind_drct,min_rh,avg_rh,max_rh,snow_in,snowd_in,min_feel,avg_feel,max_feel,max_wind_speed_kts,climo_high_f,climo_low_f
0,OOA,2015-01-01,34.88,13.1,18.5,6.62,NaN,10.909408,244.36891,48.8897,69.056470,81.6625,NaN,NaN,-3.505770,10.060907,24.5662,18.0,32.9,15.6
1,ORC,2015-01-01,30.20,8.6,24.8,3.20,0.01,6.376307,253.90510,63.0255,75.334854,86.1759,NaN,NaN,-3.344350,13.669069,30.2000,15.0,27.2,10.5
2,AWG,2015-01-01,35.60,15.8,19.4,8.60,NaN,11.317073,242.09450,47.5111,67.679660,79.4681,NaN,NaN,-0.221851,12.198088,26.5835,17.0,31.0,13.8
3,CSQ,2015-01-01,33.80,10.4,19.4,5.00,NaN,9.979095,245.56772,54.5910,69.786600,85.0468,NaN,NaN,-4.830110,9.026784,24.7411,17.0,31.6,12.5
4,EBS,2015-01-01,32.00,10.4,23.0,5.00,NaN,10.198607,252.59528,63.7541,76.983210,92.6853,NaN,NaN,-7.222420,9.500152,22.9681,17.0,27.1,9.0


## Step 2 — Fix types & enforce the join key

Parse `day` to datetime and coerce every measurement column to numeric. Rows
with no `station` or no parseable `day` can never join onto a water-quality
sample, so they are dropped. Finally we **guarantee one row per
`(station, day)`** (averaging any exact duplicates): the merge step does a
left-join on this key, and duplicate keys would silently multiply
water-quality rows.

In [3]:
df["day"] = pd.to_datetime(df["day"], errors="coerce")

NUM_COLS = [
    "max_temp_f", "min_temp_f", "max_dewpoint_f", "min_dewpoint_f",
    "precip_in", "avg_wind_speed_kts", "avg_wind_drct",
    "min_rh", "avg_rh", "max_rh",
    "snow_in", "snowd_in", "min_feel", "avg_feel", "max_feel",
    "max_wind_speed_kts", "climo_high_f", "climo_low_f",
]
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows that can never join (missing key), reporting the loss.
before = len(df)
df = df[df["station"].notna() & df["day"].notna()].copy()
print(f"Dropped rows with missing station/day: {before - len(df):,}  ({before:,} -> {len(df):,})")

# Enforce uniqueness of the join key.
n_dupes = df.duplicated(["station", "day"]).sum()
if n_dupes:
    print(f"Collapsing {n_dupes:,} duplicate (station, day) rows via mean")
    df = df.groupby(["station", "day"], as_index=False)[NUM_COLS].mean()
else:
    print("(station, day) is already unique — no collapsing needed")
assert not df.duplicated(["station", "day"]).any()

Dropped rows with missing station/day: 0  (221,559 -> 221,559)
(station, day) is already unique — no collapsing needed


## Step 3 — Range validation

All physical bounds live in one declarative dict. Values outside their bound are
**nulled** (not dropped — a single bad humidity reading shouldn't cost us the
day's temperature). Bounds are deliberately generous Iowa extremes; the goal is
to catch sensor garbage (e.g. a 600°F "feels-like"), not to trim the tails.

In [4]:
# column : (low, high) physically plausible bounds
RANGES = {
    "max_temp_f": (-60, 130), "min_temp_f": (-60, 130),
    "max_dewpoint_f": (-60, 100), "min_dewpoint_f": (-60, 100),
    "min_feel": (-80, 140), "avg_feel": (-80, 140), "max_feel": (-80, 140),
    "climo_high_f": (-60, 130), "climo_low_f": (-60, 130),
    "min_rh": (0, 100), "avg_rh": (0, 100), "max_rh": (0, 100),
    "avg_wind_drct": (0, 360),
    "avg_wind_speed_kts": (0, 150), "max_wind_speed_kts": (0, 250),
    "precip_in": (0, 30), "snow_in": (0, 60), "snowd_in": (0, 200),
}

total_nulled = 0
for col, (lo, hi) in RANGES.items():
    bad = df[col].notna() & ~df[col].between(lo, hi)
    if bad.sum():
        print(f"{col:<20} nulled {int(bad.sum()):>4} value(s) outside [{lo}, {hi}]")
    df.loc[bad, col] = np.nan
    total_nulled += int(bad.sum())
print(f"\nTotal out-of-range values nulled: {total_nulled:,}")

max_temp_f           nulled   21 value(s) outside [-60, 130]
min_temp_f           nulled    6 value(s) outside [-60, 130]
max_dewpoint_f       nulled    5 value(s) outside [-60, 100]
min_dewpoint_f       nulled   15 value(s) outside [-60, 100]
min_feel             nulled    5 value(s) outside [-80, 140]
avg_feel             nulled    7 value(s) outside [-80, 140]
max_feel             nulled   53 value(s) outside [-80, 140]
max_wind_speed_kts   nulled    4 value(s) outside [0, 250]

Total out-of-range values nulled: 116


## Step 4 — Cross-field consistency

A day's recorded minimum cannot exceed its maximum. Where it does, both ends are
recording errors, so we null the pair rather than trust either.

In [5]:
MIN_MAX_PAIRS = [
    ("min_temp_f", "max_temp_f"),
    ("min_dewpoint_f", "max_dewpoint_f"),
    ("min_rh", "max_rh"),
    ("min_feel", "max_feel"),
]
for lo_col, hi_col in MIN_MAX_PAIRS:
    bad = df[lo_col].notna() & df[hi_col].notna() & (df[lo_col] > df[hi_col])
    if bad.sum():
        print(f"{lo_col} > {hi_col}: nulled {int(bad.sum())} inconsistent pair(s)")
    df.loc[bad, [lo_col, hi_col]] = np.nan

min_temp_f > max_temp_f: nulled 1 inconsistent pair(s)
min_rh > max_rh: nulled 2 inconsistent pair(s)


## Step 5 — Missing values

- **Snow & snow depth** are *event* variables: the daily feed reports them only
  when snow occurs, so a blank means 0. We fill them with 0 (matching the
  original notebook).
- **Precipitation is left `NaN`.** Its 53% missingness is driven by *station
  instrumentation*, not by dry days — about half the stations report precip only
  ~20% of the time (no precip sensor) while the rest report it nearly always.
  Filling 0 would fabricate "no rain" for those stations and bias any
  precip-dependent model; dropping the rows (the old behaviour) would throw away
  their temperature/humidity/wind data. Leaving `NaN` lets the modeling step
  impute or branch on availability.
- All other measurements keep their `NaN`s; per-column gaps are handled at
  modeling time, and a gap in one field shouldn't discard the whole row.

In [6]:
df["snow_in"] = df["snow_in"].fillna(0)
df["snowd_in"] = df["snowd_in"].fillna(0)

miss = df[NUM_COLS].isna().sum()
miss_pct = (miss / len(df) * 100).round(1)
print("Missing values per column (count, %):")
print(pd.concat([miss.rename("missing"), miss_pct.rename("pct")], axis=1).to_string())

Missing values per column (count, %):
                    missing   pct
max_temp_f             4359   2.0
min_temp_f             4308   1.9
max_dewpoint_f         4507   2.0
min_dewpoint_f         4504   2.0
precip_in            117564  53.1
avg_wind_speed_kts     3030   1.4
avg_wind_drct          3075   1.4
min_rh                 4555   2.1
avg_rh                 4706   2.1
max_rh                 4730   2.1
snow_in                   0   0.0
snowd_in                  0   0.0
min_feel               4701   2.1
avg_feel               3082   1.4
max_feel               4751   2.1
max_wind_speed_kts     4607   2.1
climo_high_f              0   0.0
climo_low_f               0   0.0


## Step 6 — Sanity check

Confirm the headline fields now sit in sensible ranges and that we retained the
rows the old pipeline was discarding.

In [7]:
print(f"Rows retained: {len(df):,} of {n_raw:,} raw "
      f"({len(df) / n_raw:.0%}) — old pipeline kept 103,803 ({103803 / n_raw:.0%})")
print(f"Date range: {df['day'].min().date()} → {df['day'].max().date()}\n")
df[["max_temp_f", "min_temp_f", "precip_in", "avg_rh", "avg_wind_drct"]].describe().round(2)

Rows retained: 221,559 of 221,559 raw (100%) — old pipeline kept 103,803 (47%)
Date range: 2015-01-01 → 2024-12-31



,max_temp_f,min_temp_f,precip_in,avg_rh,avg_wind_drct
count,217200.00,217251.00,103995.00,216853.00,218484.00
mean,60.35,40.15,0.15,74.39,196.08
std,22.15,20.32,0.39,13.89,99.65
min,-33.50,-36.80,0.00,1.00,0.00
25%,42.80,26.60,0.00,65.51,127.70
50%,64.00,41.00,0.00,75.41,190.89
75%,80.00,58.00,0.12,84.47,289.86
max,129.20,98.60,30.00,100.00,360.00


## Step 7 — Save

In [8]:
out_file = CLEAN_DIR / "isu-climate-clean.csv"
df.to_csv(out_file, index=False)
print(f"Saved {len(df):,} station-day rows -> {out_file}")

Saved 221,559 station-day rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/climate/isu-climate-clean.csv
